# 02 — Byte-Level BPE Tokenizer From Scratch

This notebook builds a **Byte-Level BPE tokenizer from scratch**.

The goal is not to use an existing tokenizer library, but to understand:

- Why language models need tokenizers
- Why word-level tokenization is limited
- What subword tokenization solves
- How Byte Pair Encoding (BPE) works
- Why byte-level BPE starts from 256 possible byte values
- How BPE learns merge rules from a corpus
- How text is converted into token IDs
- How token IDs are decoded back into text
- How vocabulary size affects tokenization
- How tokenization changes the number of tokens seen during GPT training

## Learning Pipeline

```text
Raw Text
   ↓
UTF-8 Encoding
   ↓
Bytes
   ↓
Initial Byte Vocabulary
   ↓
Count Adjacent Token Pairs
   ↓
Merge Most Frequent Pair
   ↓
Repeat
   ↓
Learned Vocabulary + Merge Rules
   ↓
Encode Text → Token IDs
   ↓
Decode Token IDs → Text

In [ ]:
words = [
    "play",
    "played",
    "playing",
    "player",
    "plays",
    "replay",
    "replayed",
    "replaying"
]


def count_pairs(words):
    pair_counts = {}

    for word in words:
        for pair in zip(word, word[1:]):
            pair_counts[pair] = pair_counts.get(pair, 0) + 1

    return pair_counts


pair_counts = count_pairs(words)

print(pair_counts)

{('p', 'l'): 8, ('l', 'a'): 8, ('a', 'y'): 8, ('y', 'e'): 3, ('e', 'd'): 2, ('y', 'i'): 2, ('i', 'n'): 2, ('n', 'g'): 2, ('e', 'r'): 1, ('y', 's'): 1, ('r', 'e'): 3, ('e', 'p'): 3}


In [67]:
def merge_pair(tokens, pair):
    merged = []
    i = 0

    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merged.append(tokens[i] + tokens[i + 1])
            i += 2
        else:
            merged.append(tokens[i])
            i += 1

    return merged

pair = ('p', 'l')

for word in words:
    merged_words = merge_pair(word, list(pair_counts)[0])

    print(f'{word} --> {merged_words}')


play --> ['pl', 'a', 'y']
played --> ['pl', 'a', 'y', 'e', 'd']
playing --> ['pl', 'a', 'y', 'i', 'n', 'g']
player --> ['pl', 'a', 'y', 'e', 'r']
plays --> ['pl', 'a', 'y', 's']
replay --> ['r', 'e', 'pl', 'a', 'y']
replayed --> ['r', 'e', 'pl', 'a', 'y', 'e', 'd']
replaying --> ['r', 'e', 'pl', 'a', 'y', 'i', 'n', 'g']


In [90]:
words = [
    "play",
    "played",
    "playing",
    "player",
    "plays",
    "replay",
    "replayed",
    "replaying"
]

for i in range(9):
    pair_counts = count_pairs(words)
    pair = max(pair_counts, key=pair_counts.get)
    print(f'\nPair: {pair}, Count: {pair_counts[pair]}')

    words = [merge_pair(word, pair) for word in words]

    # print(f'{word} --> {merged_words}')

    for word in words:
        print(word)



Pair: ('p', 'l'), Count: 8
['pl', 'a', 'y']
['pl', 'a', 'y', 'e', 'd']
['pl', 'a', 'y', 'i', 'n', 'g']
['pl', 'a', 'y', 'e', 'r']
['pl', 'a', 'y', 's']
['r', 'e', 'pl', 'a', 'y']
['r', 'e', 'pl', 'a', 'y', 'e', 'd']
['r', 'e', 'pl', 'a', 'y', 'i', 'n', 'g']

Pair: ('pl', 'a'), Count: 8
['pla', 'y']
['pla', 'y', 'e', 'd']
['pla', 'y', 'i', 'n', 'g']
['pla', 'y', 'e', 'r']
['pla', 'y', 's']
['r', 'e', 'pla', 'y']
['r', 'e', 'pla', 'y', 'e', 'd']
['r', 'e', 'pla', 'y', 'i', 'n', 'g']

Pair: ('pla', 'y'), Count: 8
['play']
['play', 'e', 'd']
['play', 'i', 'n', 'g']
['play', 'e', 'r']
['play', 's']
['r', 'e', 'play']
['r', 'e', 'play', 'e', 'd']
['r', 'e', 'play', 'i', 'n', 'g']

Pair: ('play', 'e'), Count: 3
['play']
['playe', 'd']
['play', 'i', 'n', 'g']
['playe', 'r']
['play', 's']
['r', 'e', 'play']
['r', 'e', 'playe', 'd']
['r', 'e', 'play', 'i', 'n', 'g']

Pair: ('r', 'e'), Count: 3
['play']
['playe', 'd']
['play', 'i', 'n', 'g']
['playe', 'r']
['play', 's']
['re', 'play']
['re', 'pl

In [92]:
words = [
    "play",
    "played",
    "playing",
    "player",
    "plays",
    "replay",
    "replayed",
    "replaying"
]

# Initial vocabulary = unique characters
vocab = set("".join(words))

# Store the merge rules learned by BPE
learned_merges = []

for i in range(9):
    pair_counts = count_pairs(words)
    pair = max(pair_counts, key=pair_counts.get)

    # Create the new token
    merged_token = "".join(pair)

    # Store the learned merge
    learned_merges.append((pair, merged_token))

    # Add the new token to the vocabulary
    vocab.add(merged_token)

    print(f"Merge {i + 1}: {pair} → {merged_token}")

    words = [merge_pair(word, pair) for word in words]

print("\nFinal vocabulary:")
print(sorted(vocab))

print("\nVocabulary size:", len(vocab))

print("\nLearned merge rules:")
for pair, token in learned_merges:
    print(f"{pair} → {token}")

Merge 1: ('p', 'l') → pl
Merge 2: ('pl', 'a') → pla
Merge 3: ('pla', 'y') → play
Merge 4: ('play', 'e') → playe
Merge 5: ('r', 'e') → re
Merge 6: ('playe', 'd') → played
Merge 7: ('play', 'i') → playi
Merge 8: ('playi', 'n') → playin
Merge 9: ('playin', 'g') → playing

Final vocabulary:
['a', 'd', 'e', 'g', 'i', 'l', 'n', 'p', 'pl', 'pla', 'play', 'playe', 'played', 'playi', 'playin', 'playing', 'r', 're', 's', 'y']

Vocabulary size: 20

Learned merge rules:
('p', 'l') → pl
('pl', 'a') → pla
('pla', 'y') → play
('play', 'e') → playe
('r', 'e') → re
('playe', 'd') → played
('play', 'i') → playi
('playi', 'n') → playin
('playin', 'g') → playing


In [ ]:
# Exercise:
# Use the merge rules learned from our toy corpus to tokenize a word that was NOT present in the training corpus.
#
# Training words:
# play, played, playing, player, plays, replay, replayed, replaying
#
# Try an unseen word:
# "playful"
#
# Goal:
# See how BPE reuses learned subword tokens instead of requiring
# "playful" to exist as a complete token in the training vocabulary.


learned_merges = [
    (("p", "l"), "pl"),
    (("pl", "a"), "pla"),
    (("pla", "y"), "play"),
    (("play", "e"), "playe"),
    (("r", "e"), "re"),
    (("playe", "d"), "played"),
    (("play", "i"), "playi"),
    (("playi", "n"), "playin"),
    (("playin", "g"), "playing"),
]


def apply_merges(tokens, learned_merges):
    tokens = list(tokens)

    for pair, merged_token in learned_merges:
        new_tokens = []
        i = 0

        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and (tokens[i], tokens[i + 1]) == pair
            ):
                new_tokens.append(merged_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens


word = "playful"

tokens = list(word)

print("Original:", tokens)

tokens = apply_merges(tokens, learned_merges)

print("After BPE:", tokens)

Original: ['p', 'l', 'a', 'y', 'f', 'u', 'l']
After BPE: ['play', 'f', 'u', 'l']


### Observation

The word `playful` was not present in the training corpus.

Instead of producing an unknown token, BPE reused the learned token `play` and
represented the remaining characters separately.

This demonstrates one of the main benefits of subword tokenization:
unseen words can still be represented using previously learned pieces.